# 05 · Tổng hợp & Báo cáo

## Meta
- **Ngày chạy:** 2026-11-23 · **Deadline:** 20/11/2026 (đã qua)
- **Đầu vào chung:** `data/processed/parkinsons_v1.csv` (canonical v1)
- **Nguồn số liệu:** `results/pipeline_summary.json`, `results/m05_f1_cv.csv`, `results/m06_cluster_ari.csv`, `results/m06_dbscan_grid.csv`, `results/quality_report.md`
- **Tổng hợp:** M06 · **Gate nhất quán:** M01 (S-T2)


## Bước 1 · Tổng hợp thông tin dataset (M02/M03)
Đọc canonical v1 và in các chỉ báo đã xác minh: kích thước, nhãn, NaN/trùng, candidate outlier (IQR 1.5) và độ lệch.

In [1]:
from pathlib import Path
import pandas as pd

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/parkinsons_v1.csv').exists()), None)
assert root is not None, 'Thieu canonical CSV (S-T1)'
df = pd.read_csv(root / 'data' / 'processed' / 'parkinsons_v1.csv')
X = df.drop(columns=['status'])
q1, q3 = X.quantile(0.25), X.quantile(0.75)
iqr = q3 - q1
mask = (X < q1 - 1.5 * iqr) | (X > q3 + 1.5 * iqr)
out = mask.sum().sort_values(ascending=False)
print('shape:', df.shape)
print('label counts:', df['status'].value_counts().sort_index().to_dict())
print('NaN:', int(df.isna().sum().sum()), '| duplicate rows:', int(df.duplicated().sum()))
print('features with candidate outliers:', int((out > 0).sum()), '/', len(X.columns))
print('rows with >=1 outlier:', int(mask.any(axis=1).sum()), '/', len(df))
print('top outliers:', {k: int(v) for k, v in out.head(5).items()})
print('|skew|>1:', int((X.skew().abs() > 1).sum()), '/', len(X.columns))


shape: (195, 23)
label counts: {0: 48, 1: 147}
NaN: 0 | duplicate rows: 0
features with candidate outliers: 19 / 22
rows with >=1 outlier: 47 / 195
top outliers: {'NHR': 19, 'MDVP:PPQ': 15, 'MDVP:RAP': 14, 'Jitter:DDP': 14, 'MDVP:Jitter(%)': 14}
|skew|>1: 14 / 22


## Bước 2 · Kết quả PCA (M04)

In [2]:
import json

summary = json.loads((root / 'results' / 'pipeline_summary.json').read_text(encoding='utf-8'))
print('PCA:', summary['pca'])


PCA: {'pc1': 0.589, 'pc2': 0.113, 'pc1_pc2_total': 0.702, 'n_components_for_90pct': 6, 'cum_variance_at_90pct': 0.9161}


## Bước 3 · Kết quả phân lớp (M05)

In [3]:
f1 = pd.read_csv(root / 'results' / 'm05_f1_cv.csv')
print(f1[['model', 'f1_macro_test', 'f1_weighted_test', 'f1_macro_train']].to_string(index=False))
means = f1['f1_macro_test'].str.split(' ± ').str[0].astype(float)
best = f1.loc[means.idxmax(), 'model']
print('Cao nhất (macro):', best, '| ghi chú: không kết luận mô hình tốt nhất tuyệt đối')


       model   f1_macro_test f1_weighted_test  f1_macro_train
         KNN 0.8757 ± 0.0742  0.9073 ± 0.0578          0.9647
         SVM 0.8904 ± 0.0804  0.9199 ± 0.0603          0.9787
RandomForest 0.8737 ± 0.1171  0.9103 ± 0.0824          1.0000
Cao nhất (macro): SVM | ghi chú: không kết luận mô hình tốt nhất tuyệt đối


## Bước 4 · Kết quả phân cụm + ARI (M06)

In [4]:
ari = pd.read_csv(root / 'results' / 'm06_cluster_ari.csv')
print(ari[['K', 'silhouette', 'kich_thuoc_cum', 'ARI']].to_string(index=False))
grid = pd.read_csv(root / 'results' / 'm06_dbscan_grid.csv')
print('DBSCAN grid ARI range:', round(grid['ARI'].min(), 4), '..', round(grid['ARI'].max(), 4))


                    K  silhouette                        kich_thuoc_cum     ARI
                    2      0.4373                             [165, 30] -0.0978
                    3      0.2763                         [106, 77, 12]  0.0582
                    4      0.2358                       [91, 77, 23, 4]  0.0583
                    5      0.2048                   [73, 64, 32, 22, 4]  0.1000
                    6      0.2138               [60, 51, 30, 28, 22, 4]  0.1058
                    7      0.2094            [54, 51, 31, 24, 23, 8, 4]  0.0882
                    8      0.2201       [53, 35, 32, 23, 23, 15, 10, 4]  0.0313
                    9      0.2265     [52, 39, 25, 24, 21, 13, 9, 8, 4]  0.0574
                   10      0.1896 [38, 35, 26, 23, 22, 18, 12, 9, 8, 4]  0.0481
DBSCAN(eps=3.37,ms=5)         NaN                     [183] (nhiễu: 12) -0.0639
DBSCAN grid ARI range: -0.0744 .. 0.1625


## Bước 5 · Nhận xét
- Phân lớp: F-score macro **0.87–0.89** (10-fold CV); SVM cao nhất **0.8904 ± 0.0804**, nhưng chênh lệch giữa mô hình nhỏ so với biến động giữa fold.
- PCA: 2 thành phần giữ **70.2%**; 6 thành phần đạt **91.61%**.
- Phân cụm: K-Means chọn K=2 (silhouette 0.4373) nhưng **ARI âm**; DBSCAN ARI **−0.0639** → cấu trúc không gian đặc trưng không tách theo `status`.


## Bước 6 · Hạn chế
- Mẫu nhỏ (195), lệch nhãn 147/48; ~32 chủ thể có thể nằm ở cả train/test.
- PCA 2D chỉ giữ 70.2% phương sai; phân cụm ARI thấp; DBSCAN 1 cụm + 12 nhiễu.
- Phiên bản thư viện khác nhau giữa một số lần chạy (chi tiết trong `results/`).


## Bước 7 · Kết luận
Quy trình hoàn chỉnh trên một canonical CSV duy nhất: dữ liệu sạch, đã chuẩn hoá; **phân lớp đạt F-score macro 0.87–0.89**; **PCA** giảm chiều hiệu quả; **phân cụm** xác nhận không có cấu trúc cụm tự nhiên khớp nhãn bệnh.

Tổng hợp đầy đủ: `reports/summary.md` · Pipeline: `python -m src.pipeline`.
